In [ ]:
import pandas as pd

lanc = pd.read_csv("../data/lancamentos_raw.csv", parse_dates=["data"])
ext = pd.read_csv("../data/extrato_raw.csv", parse_dates=["data"])

print("Lançamentos:", lanc.shape)
print("Extrato:", ext.shape)
lanc.head()

In [ ]:
print("--- Valores vazios nos lançamentos ---")
print(lanc.isnull().sum())

print("\n--- Linhas 100% iguais nos lançamentos ---")
print(lanc.duplicated().sum())

print("\n--- IDs repetidos nos lançamentos ---")
print(lanc["id_lancamento"].duplicated().sum())

print("\n--- Linhas 100% iguais no extrato ---")
print(ext.duplicated().sum())

# 1) Remover lançamentos repetidos (mesmo ID)
lanc = lanc.drop_duplicates(subset="id_lancamento")

# 2) Padronizar o texto da contraparte (tirar espaços e ajustar maiúsculas)
lanc["contraparte"] = lanc["contraparte"].str.strip().str.title()

# 3) Categoria vazia: não apagar, classificar como "Não classificado"
lanc["categoria"] = lanc["categoria"].fillna("Não classificado")

print(lanc.shape)
print(lanc["categoria"].value_counts())

## Decisões
- Removi os IDs repetidos, pois eram cópias exatas
- Mantive os lançamentos sem categoria como "Não classificado" para não perder valores

In [ ]:
# Padronizar a descrição
ext["descricao"] = ext["descricao"].str.strip().str.upper()

# Sinalizar (e NÃO apagar) possíveis duplicidades
ext["possivel_duplicidade"] = ext.duplicated(
    subset=["data", "descricao", "tipo", "valor"], keep=False
)

print(ext["possivel_duplicidade"].sum(), "linhas suspeitas")
ext[ext["possivel_duplicidade"]].sort_values(["data", "descricao"]).head(6)

In [ ]:
# Fluxo do extrato por mês (crédito entra, débito sai)
ext["mes"] = ext["data"].dt.to_period("M")
ext["valor_sinal"] = ext["valor"].where(ext["tipo"] == "Crédito", -ext["valor"])

print(ext.groupby("mes")["valor_sinal"].sum())

# Total dos lançamentos por tipo e categoria
lanc.groupby(["tipo", "categoria"])["valor"].sum().sort_values(ascending=False)

In [ ]:
lanc.to_csv("../data/lancamentos_tratado.csv", index=False)
ext.to_csv("../data/extrato_tratado.csv", index=False)
print("Salvo!")

## Problemas encontrados
- 15 lançamentos com ID repetido
- 20 lançamentos sem categoria
- Textos com espaços e maiúsculas inconsistentes

In [ ]:
%pip install sqlalchemy psycopg2-binary

In [ ]:
from getpass import getpass
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

banco = "Portfolio"   # P maiúsculo, igual ao pgAdmin
porta = 5432

senha = getpass("Senha do postgres: ")
print("Caracteres digitados:", len(senha))

url = URL.create(
    "postgresql+psycopg",
    username="postgres",
    password=senha,
    host="localhost",
    port=porta,
    database=banco,
)
engine = create_engine(url)

try:
    with engine.connect() as conn:
        print("Conectou!")
except Exception as e:
    print(e.orig if hasattr(e, "orig") else e)

In [ ]:
%pip install "psycopg[binary]"

In [ ]:
from getpass import getpass
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

banco = "Portfolio"   # P maiúsculo, igual ao pgAdmin
porta = 5432

senha = getpass("Senha do postgres: ")
print("Caracteres digitados:", len(senha))

url = URL.create(
    "postgresql+psycopg",
    username="postgres",
    password=senha,
    host="localhost",
    port=porta,
    database=banco,
)
engine = create_engine(url)

try:
    with engine.connect() as conn:
        print("Conectou!")
except Exception as e:
    print(e.orig if hasattr(e, "orig") else e)

In [ ]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS lancamentos (
            id_lancamento INT PRIMARY KEY,
            data          DATE,
            tipo          VARCHAR(10),
            categoria     VARCHAR(50),
            contraparte   VARCHAR(150),
            valor         NUMERIC(12,2)
        )
    """))
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS extrato (
            id_extrato           INT PRIMARY KEY,
            data                 DATE,
            descricao            VARCHAR(200),
            tipo                 VARCHAR(10),
            valor                NUMERIC(12,2),
            possivel_duplicidade BOOLEAN
        )
    """))
    conn.execute(text("TRUNCATE lancamentos, extrato"))

print("Tabelas prontas!")

In [ ]:
lanc[["id_lancamento", "data", "tipo", "categoria", "contraparte", "valor"]] \
    .to_sql("lancamentos", engine, if_exists="append", index=False)

ext[["id_extrato", "data", "descricao", "tipo", "valor", "possivel_duplicidade"]] \
    .to_sql("extrato", engine, if_exists="append", index=False)

print("Carregado!")

In [ ]:
# Lançamentos
lanc = lanc.drop_duplicates(subset="id_lancamento")
lanc["contraparte"] = lanc["contraparte"].str.strip().str.title()
lanc["categoria"] = lanc["categoria"].fillna("Não classificado")

# Extrato
ext["descricao"] = ext["descricao"].str.strip().str.upper()
ext["possivel_duplicidade"] = ext.duplicated(
    subset=["data", "descricao", "tipo", "valor"], keep=False
)

print("Lançamentos:", lanc.shape)
print("Extrato:", ext.shape)
print("IDs repetidos:", lanc["id_lancamento"].duplicated().sum())

In [ ]:
import pandas as pd
from sqlalchemy import text

# 1) Garantir que as tabelas existem e estão vazias
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS lancamentos (
            id_lancamento INT PRIMARY KEY,
            data          DATE,
            tipo          VARCHAR(10),
            categoria     VARCHAR(50),
            contraparte   VARCHAR(150),
            valor         NUMERIC(12,2)
        )
    """))
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS extrato (
            id_extrato           INT PRIMARY KEY,
            data                 DATE,
            descricao            VARCHAR(200),
            tipo                 VARCHAR(10),
            valor                NUMERIC(12,2),
            possivel_duplicidade BOOLEAN
        )
    """))
    conn.execute(text("TRUNCATE lancamentos, extrato"))

# 2) Carregar os dados
lanc[["id_lancamento", "data", "tipo", "categoria", "contraparte", "valor"]] \
    .to_sql("lancamentos", engine, if_exists="append", index=False)

ext[["id_extrato", "data", "descricao", "tipo", "valor", "possivel_duplicidade"]] \
    .to_sql("extrato", engine, if_exists="append", index=False)

# 3) Conferir
print(pd.read_sql("SELECT COUNT(*) AS lancamentos FROM lancamentos", engine))
print(pd.read_sql("SELECT COUNT(*) AS extrato FROM extrato", engine))